In [ ]:
# MusicMatch Database Integration
from database_helper import MusicMatchDB

# Initialize database connection
try:
    db = MusicMatchDB()
    print("✅ Database connection successful!")
    
    # Get current user and sync to database
    user_data = sp.current_user()
    db_user = db.create_or_update_user(user_data)
    user_id = db_user["id"]
    
    print(f"✅ User synced to database!")
    print(f"📱 User: {user_data['display_name']}")
    print(f"🆔 Database ID: {user_id}")
    
except Exception as e:
    print(f"❌ Database connection failed: {e}")
    print("Make sure you've set up Supabase and added credentials to .env file")
    db = None
    user_id = None

In [1]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
scope = "user-library-read user-read-currently-playing app-remote-control user-modify-playback-state user-read-playback-state user-read-recently-played playlist-read-private playlist-modify-public playlist-modify-private user-follow-read user-follow-modify user-top-read user-read-playback-position user-read-email user-read-private user-read-playback-position user-read-recently-played user-library-modify user-library-read playlist-read-collaborative user-read-private user-read-email user-read-playback-state user-modify-playback-state user-read-currently-playing user-read-recently-played user-library-modify user-library-read user-follow-read user-follow-modify user-top-read playlist-read-private playlist-modify-public playlist-modify-private playlist-read-collaborative"

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=os.getenv("CLIENT_ID"),
    client_secret=os.getenv("CLIENT_SECRET"),
    redirect_uri=os.getenv("REDIRECT_URI"),
    scope=scope
    ),
    requests_timeout=20
)

### **Current User Saved Tracks**
#### Response:
- Added at
- Track:
    - Track ID
    - Track Name
    - Track Artists (object)
    - Track Album (object)
    - Track Popularity
- Artists:
    - Artist ID
    - Artist Name
- Album:
    - Album ID
    - Album Name
- Popularity



In [ ]:
tracks = sp.current_user_saved_tracks(limit=50)
for idx, item in enumerate(tracks['items']):
    print("Track: ",item['track']['name'], '--', item['track']['id'])
    print("Artists: ",[(artist['name'], artist['id'] )for artist in item['track']['artists']])
    print("Album: ",item['track']['album']['name'], '--', item['track']['album']['id'])
    print("Popularity:", item['track']['popularity'])
    print("Added at: ",item['added_at'])
    print()
    


In [ ]:
# Save user saved tracks to database
if db is not None and user_id is not None:
    try:
        # Save the saved tracks data we just retrieved
        db.save_user_saved_tracks(user_id, tracks['items'])
        
        print("✅ Saved tracks synced to database!")
        
        # Get count of saved tracks from database
        count_result = db.supabase.table("user_saved_tracks").select("count", count="exact").eq("user_id", user_id).execute()
        print(f"📊 Total saved tracks in database: {count_result.count}")
            
    except Exception as e:
        print(f"❌ Failed to save saved tracks: {e}")
else:
    print("⚠️ Skipping database save - no database connection or user ID")

### **Current User Saved Albums**
#### Response: (array of objects)
- Added at
- Album
    - Album ID
    - Album Name
    - Album Type
    - Release Date
    - Artists (array of objects)
        - Artist ID
        - Artist Name
        - Artist Type
    
    - Tracks (array of objects)
        - Track ID
        - Track Name
        - Track Popularity
        - Track Duration
        - Track Explicit
        - Track number
    
    - Popularity


In [8]:
tracks = sp.current_user_saved_albums(limit=5)
for track in tracks['items']:
    print(track['album']['name'])
    print(track['album']['artists'][0]['name'])
    print(track['album']['release_date'])
    print(track['album']['total_tracks'])
    print('---------------------------------')
    

folklore (deluxe version)
Taylor Swift
2020-08-18
17
---------------------------------
Hours Were the Birds
Adrianne Lenker
2014-01-09
10
---------------------------------
SHAKTI
Seedhe Maut
2024-08-15
4
---------------------------------
x (Deluxe Edition)
Ed Sheeran
2014-06-21
16
---------------------------------
Midnights (The Til Dawn Edition)
Taylor Swift
2023-05-26
23
---------------------------------


In [ ]:
# Save user saved albums to database
if db is not None and user_id is not None:
    try:
        # Save the saved albums data we just retrieved
        db.save_user_saved_albums(user_id, tracks['items'])
        
        print("✅ Saved albums synced to database!")
        
        # Get count of saved albums from database
        count_result = db.supabase.table("user_saved_albums").select("count", count="exact").eq("user_id", user_id).execute()
        print(f"📊 Total saved albums in database: {count_result.count}")
            
    except Exception as e:
        print(f"❌ Failed to save saved albums: {e}")
else:
    print("⚠️ Skipping database save - no database connection or user ID")